# ST FDMobileNetV1


In [ ]:
# 1)

import os
import re
import glob
import json
import time
import shutil
import zipfile
import random
import pathlib
import subprocess
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DRIVE = "/content/drive/MyDrive/bitirme_son3"


DATASET_SOURCE = os.path.join(BASE_DRIVE, "all_last7.zip")

# Output location
OUT_DRIVE_DIR = os.path.join(BASE_DRIVE, "sadece_onler", "stm32_model_zoo_clean")
os.makedirs(OUT_DRIVE_DIR, exist_ok=True)

SAVE_TEST_SET_TO_DRIVE = True
SAVE_FINAL_MODEL_TO_DRIVE = True
TEST_SET_DRIVE_DIR = os.path.join(OUT_DRIVE_DIR, "TEST_SET_USED")
FINAL_MODEL_DRIVE_DIR = os.path.join(OUT_DRIVE_DIR, "FINAL_MODEL_USED")

WORK_DIR = "/content/stm32_tomato_work"
RAW_DIR = os.path.join(WORK_DIR, "raw_dataset")
SPLIT_DIR = os.path.join(WORK_DIR, "split_dataset")

TRAIN_DIR = os.path.join(SPLIT_DIR, "train")
VAL_DIR   = os.path.join(SPLIT_DIR, "val")
TEST_DIR  = os.path.join(SPLIT_DIR, "test")

# ST repo / env / config
ST_REPO_DIR = "/content/stm32ai-modelzoo-services"
ST_IMAGE_CLASSIFICATION_DIR = os.path.join(ST_REPO_DIR, "image_classification")
ST_VENV_DIR = "/content/st_zoo"
ST_PYTHON = os.path.join(ST_VENV_DIR, "bin", "python")
CONFIG_PATH = os.path.join(
    ST_IMAGE_CLASSIFICATION_DIR,
    "config_file_examples",
    "chain_tqe_config.yaml"
)
EXP_ROOT = os.path.join(
    ST_IMAGE_CLASSIFICATION_DIR,
    "tf",
    "src",
    "experiments_outputs"
)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 120

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

MIN_IMAGES_PER_CLASS = 20

# Dataset
KEEP_CLASSES = ["flies_leaf", "healthy_leaf", "nutdef_leaf", "trips_leaf", "tuta_leaf"]

VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

print("Ayarlar hazır.")
print("DATASET_SOURCE:", DATASET_SOURCE)
print("OUT_DRIVE_DIR :", OUT_DRIVE_DIR)
print("TEST_SET_DRIVE_DIR    :", TEST_SET_DRIVE_DIR)
print("FINAL_MODEL_DRIVE_DIR :", FINAL_MODEL_DRIVE_DIR)
print("SPLIT_DIR     :", SPLIT_DIR)
print("IMG_SIZE      :", IMG_SIZE)
print("BATCH_SIZE    :", BATCH_SIZE)
print("EPOCHS        :", EPOCHS)


Mounted at /content/drive
Ayarlar hazır.
DATASET_SOURCE: /content/drive/MyDrive/bitirme_son3/all_last7.zip
OUT_DRIVE_DIR : /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean
TEST_SET_DRIVE_DIR    : /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/TEST_SET_USED
FINAL_MODEL_DRIVE_DIR : /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_MODEL_USED
SPLIT_DIR     : /content/stm32_tomato_work/split_dataset
IMG_SIZE      : 224
BATCH_SIZE    : 32
EPOCHS        : 120


In [ ]:
# 2)

def is_hidden_path(path):
    parts = pathlib.Path(path).parts
    return any(p.startswith(".") or p == "__MACOSX" for p in parts)

def list_images(folder):
    paths = []
    if not os.path.exists(folder):
        return paths

    for root, dirs, files in os.walk(folder):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__MACOSX"]
        for f in files:
            if f.lower().endswith(VALID_EXTS):
                p = os.path.join(root, f)
                if not is_hidden_path(p):
                    paths.append(p)

    return sorted(paths)

def count_immediate_class_images(root):
    """
    root/class_name/*.jpg formatını sayar.
    Recursive sayar ama class adı sadece root altındaki ilk klasörden alınır.
    """
    counts = {}
    if not os.path.isdir(root):
        return counts

    for d in sorted(os.listdir(root)):
        class_dir = os.path.join(root, d)
        if not os.path.isdir(class_dir):
            continue
        if d.startswith(".") or d == "__MACOSX":
            continue

        imgs = list_images(class_dir)
        if imgs:
            counts[d] = len(imgs)

    return counts

def find_dataset_root(start_root):
    """
    Zip açılınca bazen içeride tek bir ana klasör olur.
    En mantıklı class-folder kökünü otomatik bulur.
    """
    candidates = [start_root]

    for root, dirs, files in os.walk(start_root):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__MACOSX"]
        candidates.append(root)

    scored = []
    for c in candidates:
        counts = count_immediate_class_images(c)
        if counts:
            total = sum(counts.values())
            ncls = len(counts)
            scored.append((total, ncls, c, counts))

    if not scored:
        raise RuntimeError(
            "Dataset root bulunamadı. Beklenen yapı: class_name/*.jpg veya train/class_name/*.jpg"
        )

    scored.sort(reverse=True)
    total, ncls, root, counts = scored[0]
    print("Dataset root:", root)
    print("Bulunan class sayısı:", ncls)
    print("Toplam görüntü:", total)
    return root

def prepare_raw_dataset():
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)

    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(SPLIT_DIR, exist_ok=True)

    if not os.path.exists(DATASET_SOURCE):
        raise FileNotFoundError(f"DATASET_SOURCE bulunamadı: {DATASET_SOURCE}")

    if os.path.isfile(DATASET_SOURCE) and DATASET_SOURCE.lower().endswith(".zip"):
        print("Zip açılıyor:", DATASET_SOURCE)
        with zipfile.ZipFile(DATASET_SOURCE, "r") as zf:
            zf.extractall(RAW_DIR)
        dataset_root = find_dataset_root(RAW_DIR)

    elif os.path.isdir(DATASET_SOURCE):
        dataset_root = find_dataset_root(DATASET_SOURCE)

    else:
        raise RuntimeError("DATASET_SOURCE klasör veya .zip olmalı.")

    return dataset_root


def unique_image_filename(src_path, idx):
    """
    Aynı class içinde farklı klasörlerden gelen aynı isimli dosyalar birbirini ezmesin.
    """
    src_name = os.path.basename(src_path)
    stem = pathlib.Path(src_name).stem
    ext = pathlib.Path(src_name).suffix.lower()
    stem = re.sub(r"[^a-zA-Z0-9_\-]+", "_", stem)
    short_hash = str(abs(hash(src_path)))[:10]
    return f"{idx:06d}__{stem}__{short_hash}{ext}"

def safe_copy(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

def copy_existing_split(dataset_root):
    """
    Kaynak zaten train/val/test şeklindeyse work klasörüne temiz kopyalar.
    """
    for split in ["train", "val", "test"]:
        src_split = os.path.join(dataset_root, split)
        dst_split = os.path.join(SPLIT_DIR, split)
        shutil.copytree(src_split, dst_split, dirs_exist_ok=True)

def make_split_from_class_folders(dataset_root):
    """
    dataset_root/class_name/*.jpg yapısından 70/15/15 split oluşturur.
    """
    class_counts = count_immediate_class_images(dataset_root)

    if KEEP_CLASSES is not None:
        class_counts = {
            cls: cnt for cls, cnt in class_counts.items()
            if cls in KEEP_CLASSES
        }

    class_counts = {
        cls: cnt for cls, cnt in class_counts.items()
        if cnt >= MIN_IMAGES_PER_CLASS
    }

    if not class_counts:
        raise RuntimeError(
            "Hiç sınıf kalmadı. KEEP_CLASSES veya MIN_IMAGES_PER_CLASS ayarını kontrol et."
        )

    class_names = sorted(class_counts.keys())

    print("\nKullanılacak classlar:")
    for cls in class_names:
        print(f"{cls:25s}: {class_counts[cls]}")

    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(SPLIT_DIR, split), exist_ok=True)

    split_counts = defaultdict(Counter)

    for cls in class_names:
        cls_dir = os.path.join(dataset_root, cls)
        paths = list_images(cls_dir)

        train_paths, temp_paths = train_test_split(
            paths,
            train_size=TRAIN_RATIO,
            random_state=SEED,
            shuffle=True
        )

        val_relative = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

        val_paths, test_paths = train_test_split(
            temp_paths,
            train_size=val_relative,
            random_state=SEED,
            shuffle=True
        )

        split_map = {
            "train": train_paths,
            "val": val_paths,
            "test": test_paths,
        }

        for split, spaths in split_map.items():
            for idx, src in enumerate(spaths):
                dst_name = unique_image_filename(src, idx)
                dst = os.path.join(SPLIT_DIR, split, cls, dst_name)
                safe_copy(src, dst)
            split_counts[split][cls] = len(spaths)

    return split_counts

def get_class_names_from_split():
    class_names = sorted([
        d for d in os.listdir(TRAIN_DIR)
        if os.path.isdir(os.path.join(TRAIN_DIR, d))
    ])
    return class_names

dataset_root = prepare_raw_dataset()

has_existing_split = all(
    os.path.isdir(os.path.join(dataset_root, split))
    for split in ["train", "val", "test"]
)

if has_existing_split:
    print("\nHazır train/val/test split bulundu. Yeniden bölmeden kopyalanıyor.")
    copy_existing_split(dataset_root)
else:
    print("\nHazır split yok. Class klasörlerinden yeni split oluşturuluyor.")
    split_counts = make_split_from_class_folders(dataset_root)

class_names = get_class_names_from_split()

print("\nClass order:")
for i, cls in enumerate(class_names):
    print(i, cls)

# Split summary
summary_rows = []
for split_name, split_dir in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    print("\n" + split_name.upper())
    total = 0

    for cls in class_names:
        cnt = len(list_images(os.path.join(split_dir, cls)))
        total += cnt
        summary_rows.append({"split": split_name, "class": cls, "count": cnt})
        print(f"{cls:25s}: {cnt}")

    print("TOTAL:", total)

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUT_DRIVE_DIR, "split_summary.csv")
summary_df.to_csv(summary_csv, index=False)

with open(os.path.join(OUT_DRIVE_DIR, "classes.txt"), "w") as f:
    for c in class_names:
        f.write(c + "\n")

with open(os.path.join(OUT_DRIVE_DIR, "classes.json"), "w") as f:
    json.dump({i: c for i, c in enumerate(class_names)}, f, indent=2)


# Test set save
def copy_dir_clean(src_dir, dst_dir):
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)

def make_split_manifest(split_dir, split_name):
    rows = []
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for img_path in list_images(cls_dir):
            rows.append({
                "split": split_name,
                "class": cls,
                "filename": os.path.basename(img_path),
                "relative_path": os.path.relpath(img_path, split_dir),
                "full_path_colab": img_path,
            })
    return rows

if SAVE_TEST_SET_TO_DRIVE:
    copy_dir_clean(TEST_DIR, TEST_SET_DRIVE_DIR)

    test_manifest = pd.DataFrame(make_split_manifest(TEST_DIR, "test"))
    test_manifest_csv = os.path.join(OUT_DRIVE_DIR, "test_set_manifest.csv")
    test_manifest.to_csv(test_manifest_csv, index=False)

    print("\nTest seti Drive'a kaydedildi:")
    print(TEST_SET_DRIVE_DIR)
    print("Test manifest kaydedildi:", test_manifest_csv)
else:
    print("\nTest seti Drive'a kopyalanmadı. SAVE_TEST_SET_TO_DRIVE = False")

print("\nSPLIT_DIR hazır:", SPLIT_DIR)
print("Split özeti kaydedildi:", summary_csv)


Zip açılıyor: /content/drive/MyDrive/bitirme_son3/all_last7.zip
Dataset root: /content/stm32_tomato_work/raw_dataset/all_last
Bulunan class sayısı: 9
Toplam görüntü: 13627

Hazır split yok. Class klasörlerinden yeni split oluşturuluyor.

Kullanılacak classlar:
flies_leaf               : 1997
healthy_leaf             : 4325
nutdef_leaf              : 943
trips_leaf               : 1991
tuta_leaf                : 98

Class order:
0 flies_leaf
1 healthy_leaf
2 nutdef_leaf
3 trips_leaf
4 tuta_leaf

TRAIN
flies_leaf               : 1397
healthy_leaf             : 3027
nutdef_leaf              : 660
trips_leaf               : 1393
tuta_leaf                : 68
TOTAL: 6545

VAL
flies_leaf               : 300
healthy_leaf             : 649
nutdef_leaf              : 141
trips_leaf               : 299
tuta_leaf                : 15
TOTAL: 1404

TEST
flies_leaf               : 300
healthy_leaf             : 649
nutdef_leaf              : 142
trips_leaf               : 299
tuta_leaf               

In [ ]:
# 3) ST MODEL ZOO ENVIROMENT

REINSTALL_ST_ENV = False

if REINSTALL_ST_ENV and os.path.exists(ST_VENV_DIR):
    print("Eski ST virtualenv siliniyor:", ST_VENV_DIR)
    shutil.rmtree(ST_VENV_DIR)

if not os.path.exists(ST_REPO_DIR):
    print("Repo indiriliyor...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/STMicroelectronics/stm32ai-modelzoo-services.git", ST_REPO_DIR],
        check=True
    )
else:
    print("Repo zaten var:", ST_REPO_DIR)

if not os.path.exists(ST_PYTHON):
    print("virtualenv kuruluyor...")
    subprocess.run(["python3", "-m", "pip", "install", "-q", "--upgrade", "pip", "virtualenv"], check=True)

    py312 = shutil.which("python3.12") or shutil.which("python3")
    print("Virtualenv Python:", py312)

    subprocess.run(["python3", "-m", "virtualenv", "--download", "--python", py312, ST_VENV_DIR], check=True)

    subprocess.run([ST_PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)

    req_path = os.path.join(ST_REPO_DIR, "requirements.txt")
    subprocess.run([ST_PYTHON, "-m", "pip", "install", "-r", req_path], check=True)

    subprocess.run([
        ST_PYTHON, "-m", "pip", "install",
        "clearml", "texttable", "munch", "pyvww", "onnx", "onnxruntime", "tf2onnx"
    ], check=True)
else:
    print("ST virtualenv zaten var:", ST_VENV_DIR)

print("ST ortamı hazır.")
print("ST_PYTHON:", ST_PYTHON)


Repo indiriliyor...
virtualenv kuruluyor...
Virtualenv Python: /usr/bin/python3.12


In [ ]:
# 4)

check_code = r"""
import os
import tensorflow as tf
import keras
import clearml
import texttable
import munch
import pyvww
import onnx
import onnxruntime

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("clearml OK")
print("texttable OK")
print("munch OK")
print("pyvww OK")
print("onnx OK")
print("onnxruntime OK")
"""

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["TF_CPP_MIN_LOG_LEVEL"] = "2"

subprocess.run([ST_PYTHON, "-c", check_code], check=True, env=env)


In [ ]:
# 5) ST FDMOBILENETV1

assert os.path.exists(TRAIN_DIR), f"TRAIN_DIR yok: {TRAIN_DIR}"
assert os.path.exists(VAL_DIR),   f"VAL_DIR yok: {VAL_DIR}"
assert os.path.exists(TEST_DIR),  f"TEST_DIR yok: {TEST_DIR}"

class_names = get_class_names_from_split()
num_classes = len(class_names)

if num_classes < 2:
    raise RuntimeError("En az 2 class gerekli.")

print("Class names:")
for i, c in enumerate(class_names):
    print(i, c)

class_names_yaml = "[" + ", ".join([f'"{c}"' for c in class_names]) + "]"
project_name = f"tomato_stm32_st_fdmobilenetv1_{num_classes}class_chain_tqe"

config_text = f"""
general:
  project_name: {project_name}
  logs_dir: logs
  saved_models_dir: saved_models
  display_figures: False
  global_seed: {SEED}
  gpu_memory_limit: 12

operation_mode: chain_tqe

model:
  model_name: st_fdmobilenetv1
  input_shape: ({IMG_SIZE}, {IMG_SIZE}, 3)
  pretrained: False

dataset:
  dataset_name: custom_dataset
  class_names: {class_names_yaml}
  training_path: {TRAIN_DIR}
  validation_path: {VAL_DIR}
  validation_split:
  test_path: {TEST_DIR}
  quantization_split: 0.25

preprocessing:
  rescaling:
    scale: 1/127.5
    offset: -1
  resizing:
    aspect_ratio: fit
    interpolation: area
  color_mode: rgb

data_augmentation:
  random_contrast:
    factor: 0.25
  random_brightness:
    factor: 0.12
  random_flip:
    mode: horizontal_and_vertical
  random_translation:
    width_factor: 0.04
    height_factor: 0.04
  random_rotation:
    factor: 0.06
  random_zoom:
    width_factor: 0.04
    height_factor: 0.04

training:
  frozen_layers:
  dropout: 0.30
  batch_size: {BATCH_SIZE}
  epochs: {EPOCHS}
  optimizer:
    Adam:
      learning_rate: 0.001
  callbacks:
    ReduceLROnPlateau:
      monitor: val_accuracy
      mode: max
      factor: 0.5
      patience: 8
      min_lr: 1.0e-6
    EarlyStopping:
      monitor: val_accuracy
      mode: max
      restore_best_weights: True
      patience: 25

quantization:
  quantizer: TFlite_converter
  quantization_type: PTQ
  quantization_input_type: uint8
  quantization_output_type: float
  export_dir: quantized_models

mlflow:
  uri: ./tf/src/experiments_outputs/mlruns

hydra:
  run:
    dir: ./tf/src/experiments_outputs/${{now:%Y_%m_%d_%H_%M_%S}}
"""

os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)

with open(CONFIG_PATH, "w") as f:
    f.write(config_text)

config_drive_path = os.path.join(OUT_DRIVE_DIR, "used_config_before_run.yaml")
with open(config_drive_path, "w") as f:
    f.write(config_text)

print("\nConfig yazıldı:")
print(CONFIG_PATH)
print("Drive kopyası:", config_drive_path)

print("\nÖnemli ayarlar:")
print("project_name:", project_name)
print("model_name: st_fdmobilenetv1")
print("operation_mode: chain_tqe")
print("input_shape:", (IMG_SIZE, IMG_SIZE, 3))
print("batch_size:", BATCH_SIZE)
print("epochs:", EPOCHS)
print("class_count:", num_classes)


Class names:
0 flies_leaf
1 healthy_leaf
2 nutdef_leaf
3 trips_leaf
4 tuta_leaf

Config yazıldı:
/content/stm32ai-modelzoo-services/image_classification/config_file_examples/chain_tqe_config.yaml
Drive kopyası: /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/used_config_before_run.yaml

Önemli ayarlar:
project_name: tomato_stm32_st_fdmobilenetv1_5class_chain_tqe
model_name: st_fdmobilenetv1
operation_mode: chain_tqe
input_shape: (224, 224, 3)
batch_size: 32
epochs: 120
class_count: 5


In [ ]:
# 6) TRAINING + QUANTIZATION + TEST

log_path = os.path.join(OUT_DRIVE_DIR, "training_quant_eval_log.txt")

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["TF_CPP_MIN_LOG_LEVEL"] = "1"
env["HYDRA_FULL_ERROR"] = "1"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    ST_PYTHON,
    "-u",
    "stm32ai_main.py",
    "--config-path", "./config_file_examples",
    "--config-name", "chain_tqe_config",
]

print("Çalıştırılan komut:")
print(" ".join(cmd))
print("\nLog dosyası:", log_path)
print("=" * 80)

with open(log_path, "w", buffering=1) as log_file:
    process = subprocess.Popen(
        cmd,
        cwd=ST_IMAGE_CLASSIFICATION_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

print("\nEğitim + quantization + test tamamlandı.")
print("Log kaydedildi:", log_path)


Streaming output truncated to the last 5000 lines.
108/205 ━━━━━━━━━━━━━━━━━━━━ 25s 267ms/step - accuracy: 0.9215 - loss: 0.2077
109/205 ━━━━━━━━━━━━━━━━━━━━ 25s 266ms/step - accuracy: 0.9216 - loss: 0.2075
110/205 ━━━━━━━━━━━━━━━━━━━━ 25s 266ms/step - accuracy: 0.9217 - loss: 0.2073
111/205 ━━━━━━━━━━━━━━━━━━━━ 24s 266ms/step - accuracy: 0.9217 - loss: 0.2070
112/205 ━━━━━━━━━━━━━━━━━━━━ 24s 266ms/step - accuracy: 0.9218 - loss: 0.2068
113/205 ━━━━━━━━━━━━━━━━━━━━ 24s 265ms/step - accuracy: 0.9219 - loss: 0.2066
114/205 ━━━━━━━━━━━

In [ ]:
# 7) FLOAT MODEL TEST

eval_code = f"""
import os
import glob
import json
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE = {IMG_SIZE}
BATCH_SIZE = {BATCH_SIZE}
TEST_DIR = r"{TEST_DIR}"
EXP_ROOT = r"{EXP_ROOT}"
OUT_DIR = r"{OUT_DRIVE_DIR}"

os.makedirs(OUT_DIR, exist_ok=True)

model_files = glob.glob(os.path.join(EXP_ROOT, "**", "best_model.keras"), recursive=True)
model_files += glob.glob(os.path.join(EXP_ROOT, "**", "*.keras"), recursive=True)
model_files += glob.glob(os.path.join(EXP_ROOT, "**", "*.h5"), recursive=True)

if not model_files:
    raise RuntimeError("Model bulunamadı. Eğitim başarısız bitmiş olabilir.")

latest_model = max(model_files, key=os.path.getmtime)
print("Loaded model:", latest_model)

model = tf.keras.models.load_model(latest_model, compile=False)

raw_test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = raw_test_ds.class_names
print("Class names:", class_names)

def st_preprocess(images, labels):
    images = tf.cast(images, tf.float32)
    images = images / 127.5 - 1.0
    return images, labels

test_ds = raw_test_ds.map(st_preprocess).prefetch(tf.data.AUTOTUNE)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

test_loss, test_acc = model.evaluate(test_ds, verbose=1)

print("TEST LOSS:", test_loss)
print("TEST ACC :", test_acc)

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
)

cm = confusion_matrix(y_true, y_pred)

print("===== CLASSIFICATION REPORT =====")
print(report)

print("===== CONFUSION MATRIX =====")
print(cm)

report_path = os.path.join(OUT_DIR, "st_fdmobilenetv1_test_report.txt")
cm_path = os.path.join(OUT_DIR, "st_fdmobilenetv1_confusion_matrix.csv")
classes_path = os.path.join(OUT_DIR, "st_fdmobilenetv1_classes.json")

with open(report_path, "w") as f:
    f.write("MODEL:\\n")
    f.write(latest_model + "\\n\\n")
    f.write(f"TEST LOSS: {{test_loss}}\\n")
    f.write(f"TEST ACC : {{test_acc}}\\n\\n")
    f.write("CLASSIFICATION REPORT:\\n")
    f.write(report)
    f.write("\\nCONFUSION MATRIX:\\n")
    f.write(str(cm))

np.savetxt(cm_path, cm, delimiter=",", fmt="%d")

with open(classes_path, "w") as f:
    json.dump({{i: c for i, c in enumerate(class_names)}}, f, indent=2)

print("Test raporu kaydedildi:", report_path)
print("Confusion matrix kaydedildi:", cm_path)
print("Class json kaydedildi:", classes_path)
"""

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["TF_CPP_MIN_LOG_LEVEL"] = "1"

subprocess.run([ST_PYTHON, "-c", eval_code], check=True, env=env)


CompletedProcess(args=['/content/st_zoo/bin/python', '-c', '\nimport os\nimport glob\nimport json\nimport numpy as np\nimport tensorflow as tf\nfrom sklearn.metrics import classification_report, confusion_matrix\n\nIMG_SIZE = 224\nBATCH_SIZE = 32\nTEST_DIR = r"/content/stm32_tomato_work/split_dataset/test"\nEXP_ROOT = r"/content/stm32ai-modelzoo-services/image_classification/tf/src/experiments_outputs"\nOUT_DIR = r"/content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean"\n\nos.makedirs(OUT_DIR, exist_ok=True)\n\nmodel_files = glob.glob(os.path.join(EXP_ROOT, "**", "best_model.keras"), recursive=True)\nmodel_files += glob.glob(os.path.join(EXP_ROOT, "**", "*.keras"), recursive=True)\nmodel_files += glob.glob(os.path.join(EXP_ROOT, "**", "*.h5"), recursive=True)\n\nif not model_files:\n    raise RuntimeError("Model bulunamadı. Eğitim başarısız bitmiş olabilir.")\n\nlatest_model = max(model_files, key=os.path.getmtime)\nprint("Loaded model:", latest_model)\n\nmodel = tf.ker

In [ ]:
# 8) RESULTS

def latest_subdir(root):
    dirs = [
        d for d in glob.glob(os.path.join(root, "*"))
        if os.path.isdir(d)
    ]
    if not dirs:
        return None
    return max(dirs, key=os.path.getmtime)

def newest_file(files):
    files = [f for f in files if os.path.isfile(f)]
    if not files:
        return None
    return max(files, key=os.path.getmtime)

def find_files(root, patterns):
    out = []
    if root is None or not os.path.exists(root):
        return out
    for pat in patterns:
        out.extend(glob.glob(os.path.join(root, "**", pat), recursive=True))
    return sorted(set([f for f in out if os.path.isfile(f)]))

def copy_with_unique_name(src, dst_dir, preferred_name=None):
    os.makedirs(dst_dir, exist_ok=True)
    if preferred_name is None:
        preferred_name = os.path.basename(src)

    dst = os.path.join(dst_dir, preferred_name)
    base = pathlib.Path(preferred_name).stem
    ext = pathlib.Path(preferred_name).suffix

    k = 1
    while os.path.exists(dst):
        try:
            if os.path.getsize(src) == os.path.getsize(dst):
                return dst
        except Exception:
            pass
        dst = os.path.join(dst_dir, f"{base}_{k}{ext}")
        k += 1

    shutil.copy2(src, dst)
    return dst

latest_exp_dir = latest_subdir(EXP_ROOT)

SAVE_DIR = os.path.join(
    OUT_DRIVE_DIR,
    "FINAL_SAVED_" + time.strftime("%Y%m%d_%H%M%S")
)

os.makedirs(SAVE_DIR, exist_ok=True)

print("SAVE_DIR:", SAVE_DIR)

if latest_exp_dir is not None:
    dst_exp = os.path.join(SAVE_DIR, "latest_experiment")
    shutil.copytree(latest_exp_dir, dst_exp, dirs_exist_ok=True)
    print("En güncel experiment kopyalandı:")
    print("FROM:", latest_exp_dir)
    print("TO  :", dst_exp)
else:
    print("EXP_ROOT altında experiment bulunamadı:", EXP_ROOT)

model_search_root = latest_exp_dir if latest_exp_dir is not None else EXP_ROOT

if SAVE_FINAL_MODEL_TO_DRIVE:
    if os.path.exists(FINAL_MODEL_DRIVE_DIR):
        shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
    os.makedirs(FINAL_MODEL_DRIVE_DIR, exist_ok=True)

    copied_models = []

    float_candidates = find_files(model_search_root, ["best_model.keras", "*.keras", "*.h5"])
    best_float = newest_file(float_candidates)
    if best_float is not None:
        ext = pathlib.Path(best_float).suffix
        dst = copy_with_unique_name(best_float, FINAL_MODEL_DRIVE_DIR, "final_float_model" + ext)
        copied_models.append({"kind": "float_model", "source": best_float, "saved_to": dst})
        print("Float model Drive'a kaydedildi:", dst)
    else:
        print("Float model bulunamadı (.keras/.h5).")

    deploy_candidates = find_files(model_search_root, ["*.tflite", "*.onnx"])
    for src_model in deploy_candidates:
        dst = copy_with_unique_name(src_model, FINAL_MODEL_DRIVE_DIR)
        copied_models.append({"kind": "deploy_model", "source": src_model, "saved_to": dst})
        print("Deploy model Drive'a kaydedildi:", dst)

    for helper_file in [
        os.path.join(OUT_DRIVE_DIR, "used_config_before_run.yaml"),
        os.path.join(OUT_DRIVE_DIR, "classes.txt"),
        os.path.join(OUT_DRIVE_DIR, "classes.json"),
    ]:
        if os.path.exists(helper_file):
            dst = copy_with_unique_name(helper_file, FINAL_MODEL_DRIVE_DIR)
            copied_models.append({"kind": "helper_file", "source": helper_file, "saved_to": dst})

    model_manifest_csv = os.path.join(FINAL_MODEL_DRIVE_DIR, "model_files_manifest.csv")
    pd.DataFrame(copied_models).to_csv(model_manifest_csv, index=False)
    print("Model manifest kaydedildi:", model_manifest_csv)
else:
    print("Model dosyaları ayrıca Drive'a kopyalanmadı. SAVE_FINAL_MODEL_TO_DRIVE = False")

if os.path.exists(FINAL_MODEL_DRIVE_DIR):
    shutil.copytree(
        FINAL_MODEL_DRIVE_DIR,
        os.path.join(SAVE_DIR, "FINAL_MODEL_USED"),
        dirs_exist_ok=True
    )

if SAVE_TEST_SET_TO_DRIVE and os.path.exists(TEST_SET_DRIVE_DIR):
    shutil.copytree(
        TEST_SET_DRIVE_DIR,
        os.path.join(SAVE_DIR, "TEST_SET_USED"),
        dirs_exist_ok=True
    )
    print("Test set final pakete de kopyalandı:", os.path.join(SAVE_DIR, "TEST_SET_USED"))
elif os.path.exists(TEST_DIR):
    shutil.copytree(TEST_DIR, os.path.join(SAVE_DIR, "TEST_SET_USED"), dirs_exist_ok=True)
    print("Test set Colab splitinden final pakete kopyalandı:", os.path.join(SAVE_DIR, "TEST_SET_USED"))

for src_path in [
    CONFIG_PATH,
    os.path.join(OUT_DRIVE_DIR, "used_config_before_run.yaml"),
    os.path.join(OUT_DRIVE_DIR, "training_quant_eval_log.txt"),
    os.path.join(OUT_DRIVE_DIR, "st_fdmobilenetv1_test_report.txt"),
    os.path.join(OUT_DRIVE_DIR, "st_fdmobilenetv1_confusion_matrix.csv"),
    os.path.join(OUT_DRIVE_DIR, "st_fdmobilenetv1_classes.json"),
    os.path.join(OUT_DRIVE_DIR, "classes.txt"),
    os.path.join(OUT_DRIVE_DIR, "classes.json"),
    os.path.join(OUT_DRIVE_DIR, "split_summary.csv"),
    os.path.join(OUT_DRIVE_DIR, "test_set_manifest.csv"),
]:
    if os.path.exists(src_path):
        shutil.copy2(src_path, os.path.join(SAVE_DIR, os.path.basename(src_path)))
        print("Kopyalandı:", os.path.basename(src_path))

print("\nBulunan önemli model dosyaları:")
for ext in ["*.keras", "*.h5", "*.tflite", "*.onnx"]:
    files = glob.glob(os.path.join(SAVE_DIR, "**", ext), recursive=True)
    print(ext, ":", len(files))
    for f in files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {f}  ({size_mb:.2f} MB)")

print("\nBİTTİ. Ana final paket burada:")
print(SAVE_DIR)
print("\nTest set ayrıca burada:")
print(TEST_SET_DRIVE_DIR)
print("\nModel dosyaları ayrıca burada:")
print(FINAL_MODEL_DRIVE_DIR)


SAVE_DIR: /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_SAVED_20260602_213616
En güncel experiment kopyalandı:
FROM: /content/stm32ai-modelzoo-services/image_classification/tf/src/experiments_outputs/2026_06_02_19_13_20
TO  : /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_SAVED_20260602_213616/latest_experiment
Float model Drive'a kaydedildi: /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_MODEL_USED/final_float_model.keras
Deploy model Drive'a kaydedildi: /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_MODEL_USED/quantized_model.tflite
Model manifest kaydedildi: /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_MODEL_USED/model_files_manifest.csv
Test set final pakete de kopyalandı: /content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_SAVED_20260602_213616/TEST_SET_USED
Kopyalandı: chain_tqe_config.yaml
Kopyalandı: us

In [ ]:
# 9) ZIP

MAKE_ZIP = True

if MAKE_ZIP:
    zip_path = SAVE_DIR + ".zip"
    shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)

    print("Zip oluşturuldu:")
    print(zip_path)
    print("Boyut MB:", os.path.getsize(zip_path) / (1024 * 1024))
else:
    print("ZIP kapalı. Açmak için MAKE_ZIP = True yap.")


Zip oluşturuldu:
/content/drive/MyDrive/bitirme_son3/sadece_onler/stm32_model_zoo_clean/FINAL_SAVED_20260602_213616.zip
Boyut MB: 323.65798473358154
